# Distributed Kaggle Worker
### AI Anime Video Factory - Kaggle GPU Rendering

**Prerequisites:**
1. Create Kaggle Dataset `anime-factory-models` with 4 model files
2. Add it as input to this notebook
3. Add service account JSON as Kaggle Secret: `GDRIVE_SERVICE_ACCOUNT`
4. **Settings -> Internet -> ON** (required!)
5. Settings -> Accelerator -> GPU T4 x2

In [ ]:
# CELL 1: Install Dependencies
# IMPORTANT: Make sure Internet is ON in Settings!
!pip install -q aiohttp einops transformers safetensors accelerate pyyaml pillow scipy
!pip install -q google-auth google-api-python-client
!apt-get -qq install -y ffmpeg > /dev/null 2>&1

print('Dependencies installed!')

In [ ]:
# CELL 2: Connect to Google Drive via Service Account
# Make sure you added the secret GDRIVE_SERVICE_ACCOUNT in Kaggle settings
import json, os
from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload
from kaggle_secrets import UserSecretsClient
import io

secrets = UserSecretsClient()
sa_json = secrets.get_secret('GDRIVE_SERVICE_ACCOUNT')
sa_info = json.loads(sa_json)

SCOPES = ['https://www.googleapis.com/auth/drive']
creds = service_account.Credentials.from_service_account_info(sa_info, scopes=SCOPES)
drive_service = build('drive', 'v3', credentials=creds)

print('Google Drive API connected!')

In [ ]:
# CELL 3: Drive Sync Helpers
import os, io

def find_folder_id(name, parent_id=None):
    q = f"name='{name}' and mimeType='application/vnd.google-apps.folder' and trashed=false"
    if parent_id:
        q += f" and '{parent_id}' in parents"
    results = drive_service.files().list(q=q, fields='files(id,name)').execute()
    files = results.get('files', [])
    return files[0]['id'] if files else None

def download_file(file_id, local_path):
    request = drive_service.files().get_media(fileId=file_id)
    os.makedirs(os.path.dirname(local_path) or '.', exist_ok=True)
    with open(local_path, 'wb') as f:
        downloader = MediaIoBaseDownload(f, request)
        done = False
        while not done:
            status, done = downloader.next_chunk()

def upload_file(local_path, parent_id, filename=None):
    fname = filename or os.path.basename(local_path)
    q = f"name='{fname}' and '{parent_id}' in parents and trashed=false"
    existing = drive_service.files().list(q=q, fields='files(id)').execute().get('files', [])
    media = MediaFileUpload(local_path, resumable=True)
    if existing:
        drive_service.files().update(fileId=existing[0]['id'], media_body=media).execute()
    else:
        drive_service.files().create(body={'name': fname, 'parents': [parent_id]}, media_body=media).execute()

def list_files(folder_id):
    results = drive_service.files().list(
        q=f"'{folder_id}' in parents and trashed=false",
        fields='files(id,name,modifiedTime)'
    ).execute()
    return results.get('files', [])

FACTORY_ID = find_folder_id('AnimeFactory')
if not FACTORY_ID:
    print('AnimeFactory folder not found! Run drive_uploader.py on your PC first.')
else:
    STATE_ID = find_folder_id('state', FACTORY_ID)
    LOCKS_ID = find_folder_id('locks', STATE_ID)
    TTS_ID = find_folder_id('tts', find_folder_id('inputs', FACTORY_ID))
    SCENES_ID = find_folder_id('scenes', find_folder_id('outputs', FACTORY_ID))
    print(f'Found AnimeFactory on Drive (ID: {FACTORY_ID})')

In [ ]:
# CELL 4: Install ComfyUI + Link Models from Kaggle Dataset
import os
os.chdir('/kaggle/working')

if not os.path.exists('/kaggle/working/ComfyUI'):
    !git clone https://github.com/comfyanonymous/ComfyUI.git
    os.chdir('/kaggle/working/ComfyUI')
    !pip install -q -r requirements.txt
else:
    os.chdir('/kaggle/working/ComfyUI')

CUSTOM_NODES = '/kaggle/working/ComfyUI/custom_nodes'
if not os.path.exists(f'{CUSTOM_NODES}/ComfyUI-VideoHelperSuite'):
    os.chdir(CUSTOM_NODES)
    !git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git
    os.chdir('/kaggle/working/ComfyUI')

# Symlink models from Kaggle Dataset (instant!)
DATASET = '/kaggle/input/anime-factory-models'
MODEL_LINKS = {
    'models/checkpoints/AnythingXL_xl.safetensors': 'AnythingXL_xl.safetensors',
    'models/diffusion_models/wan2.1_t2v_1.3B_fp16.safetensors': 'wan2.1_t2v_1.3B_fp16.safetensors',
    'models/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors': 'umt5_xxl_fp8_e4m3fn_scaled.safetensors',
    'models/vae/wan_2.1_vae.safetensors': 'wan_2.1_vae.safetensors',
}

for dest, src_name in MODEL_LINKS.items():
    dest_path = os.path.join('/kaggle/working/ComfyUI', dest)
    src_path = os.path.join(DATASET, src_name)
    os.makedirs(os.path.dirname(dest_path), exist_ok=True)
    if os.path.exists(src_path) and not os.path.exists(dest_path):
        os.symlink(src_path, dest_path)
        print(f'Linked {src_name}')
    elif os.path.exists(dest_path):
        print(f'Already linked {src_name}')
    else:
        print(f'{src_name} not found in dataset!')

print('ComfyUI + models ready!')

In [ ]:
# CELL 5: Start ComfyUI Server
import subprocess, time
os.chdir('/kaggle/working/ComfyUI')

comfy_proc = subprocess.Popen(
    ['python', 'main.py', '--listen', '0.0.0.0', '--port', '8188', '--dont-print-server'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)

print('Waiting for ComfyUI to start...')
time.sleep(25)
print('ComfyUI server running!')

In [ ]:
# CELL 6: Kaggle Worker Loop
import json, os, random, time, uuid, urllib.request, subprocess, shutil

COMFYUI_URL = 'http://127.0.0.1:8188'
COMFYUI_OUTPUT = '/kaggle/working/ComfyUI/output'
LOCAL_TTS = '/kaggle/working/tts'
os.makedirs(LOCAL_TTS, exist_ok=True)

STYLE_PREFIX = 'manhwa style, webtoon art, sharp linework, vibrant colors, ultra detailed, masterpiece, best quality, '
NEGATIVE_PROMPT = 'low quality, worst quality, blurry, bad anatomy, bad proportions, deformed, ugly, 3d render, photorealistic, watermark, text, signature, extra limbs'

WORKER_ID = f'kaggle_{uuid.uuid4().hex[:8]}'
print(f'Worker ID: {WORKER_ID}')

def sync_state_from_drive():
    for fname in ['progress.json', 'master_script.json']:
        files = [f for f in list_files(STATE_ID) if f['name'] == fname]
        if files:
            download_file(files[0]['id'], f'/kaggle/working/{fname}')

def sync_state_to_drive():
    upload_file('/kaggle/working/progress.json', STATE_ID, 'progress.json')

def download_tts(scene_id):
    fname = f'{scene_id}.mp3'
    local_path = os.path.join(LOCAL_TTS, fname)
    if not os.path.exists(local_path):
        files = [f for f in list_files(TTS_ID) if f['name'] == fname]
        if files:
            download_file(files[0]['id'], local_path)
    return local_path if os.path.exists(local_path) else None

def upload_scene(local_path, scene_id):
    upload_file(local_path, SCENES_ID, f'{scene_id}.mp4')

# Main loop
sync_state_from_drive()
with open('/kaggle/working/master_script.json') as f:
    script = json.load(f)
with open('/kaggle/working/progress.json') as f:
    progress = json.load(f)

scenes_done = 0
for scene_id, status in progress['scenes'].items():
    if status != 'pending':
        continue
    idx = int(scene_id.split('_')[1]) - 1
    scene = script[idx]
    print(f'Processing: {scene_id}')

    progress['scenes'][scene_id] = 'locked'
    with open('/kaggle/working/progress.json', 'w') as f:
        json.dump(progress, f, indent=2)
    sync_state_to_drive()

    audio_path = download_tts(scene_id)
    if not audio_path:
        print(f'Missing TTS for {scene_id}, skipping')
        progress['scenes'][scene_id] = 'pending'
        continue

    # TODO: Add full ComfyUI generation logic here
    print(f'Generate visuals for {scene_id} via ComfyUI API')

    progress['scenes'][scene_id] = 'done'
    with open('/kaggle/working/progress.json', 'w') as f:
        json.dump(progress, f, indent=2)
    sync_state_to_drive()
    scenes_done += 1

print(f'Worker {WORKER_ID} completed {scenes_done} scenes!')